# 🇵🇱 RAG z Llama-PLLuM-8B-chat

**Model:** `CYFRAGOVPL/Llama-PLLuM-8B-chat`  
**Embeddingi:** `intfloat/multilingual-e5-base`  
**Baza wektorowa:** FAISS (lokalnie)  
**Kaggle:** `GPUT 4 x 2`

### Pipeline:
```
Dokumenty → chunk → embed → FAISS
Pytanie   → embed → similarity_search → top-k fragmentów
                                            ↓
                               PLLuM RAG prompt template
                                            ↓
                               Llama-PLLuM-8B-chat
                                            ↓
                               Odpowiedź z cytatami [0][1]…
```

### Przed startem:
- `Kaggle` → `Stettings` → `Accelerator` → **GPU T4 x 2**


## 1. Instalacja

In [1]:
%%capture
!pip install unsloth
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu pypdf
!pip install trl accelerate peft datasets bitsandbytes

## 2. Sprawdzenie GPU

In [2]:
!nvidia-smi

Fri Apr 17 16:05:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Indeksowanie dokumentów

**Zamień `documents` na swoje dane.**  
Możesz wgrać PDF-y przez panel po lewej (ikona folderu) lub wpisać teksty ręcznie.

In [ ]:
# ===== DOKUMENTY =====
# Opcja A: tekst wpisany ręcznie
raw_documents = [
    """
    ZUS (Zakład Ubezpieczeń Społecznych) jest państwową instytucją ubezpieczeń społecznych
    w Polsce. Zajmuje się gromadzeniem składek oraz wypłatą świadczeń takich jak emerytury,
    renty, zasiłki chorobowe i macierzyńskie. Siedziba główna ZUS znajduje się w Warszawie.
    """,
    """
    Składki na ubezpieczenie emerytalne wynoszą łącznie 19,52% podstawy wymiaru.
    Są one finansowane po połowie przez pracownika (9,76%) i pracodawcę (9,76%).
    Składki rentowe wynoszą 8% podstawy wymiaru — pracownik opłaca 1,5%, pracodawca 6,5%.
    Składka chorobowa wynosi 2,45% i jest opłacana w całości przez pracownika.
    """,
    """
    Zasiłek chorobowy przysługuje ubezpieczonemu, który stał się niezdolny do pracy
    z powodu choroby w czasie trwania ubezpieczenia. Wynosi 80% podstawy wymiaru
    zasiłku za każdy dzień niezdolności do pracy. W przypadku pobytu w szpitalu
    zasiłek wynosi 70%. Wniosek ZUS ZLA należy złożyć w ciągu 7 dni od daty wystawienia.
    """,
    """
    Emerytura z ZUS to świadczenie pieniężne mające służyć jako zabezpieczenie bytu
    na starość. Wiek emerytalny w Polsce wynosi 60 lat dla kobiet i 65 lat dla mężczyzn.
    Wysokość emerytury zależy od sumy zebranych składek oraz przewidywanego dalszego
    trwania życia. Minimalna emerytura w 2024 roku wynosi 1780,96 zł brutto.
    """,
    """
    Płatnik składek ma obowiązek zgłosić pracownika do ubezpieczeń społecznych
    w ciągu 7 dni od daty powstania obowiązku ubezpieczenia na druku ZUS ZUA.
    Miesięczne deklaracje rozliczeniowe ZUS DRA należy składać do 15. dnia
    następnego miesiąca. Składki należy opłacić do 15. dnia miesiąca za miesiąc
    poprzedni (dla osób prawnych i jednostek organizacyjnych).
    """,
]

# Opcja B: wczytaj PDF (Kaggle)
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("/kaggle/input/NAZWA-DATASETU/moj_dokument.pdf")
# pages = loader.load()
# raw_documents = [p.page_content for p in pages]
# print(f"Wczytano {len(pages)} stron z PDF")

# ===== KONIEC DOKUMENTÓW =====
print(f"Załadowano {len(raw_documents)} dokumentów.")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, 
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " "],
)
docs = [Document(page_content=t.strip()) for t in raw_documents if t.strip()]
chunks = splitter.split_documents(docs)
print(f"Podzielono na {len(chunks)} fragmentów.")

# Model embeddingów — wielojęzyczny, obsługuje polski
#    multilingual-e5-base: 278M parametrów, ~1GB VRAM
print("Ładowanie modelu embeddingów...")
embed_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

# Zapis do bazy FAISS
print("Indeksowanie...")
vectorstore = FAISS.from_documents(chunks, embed_model)
vectorstore.save_local("./pllum_rag_index")
print(f"✅ Zaindeksowano {len(chunks)} fragmentów → ./pllum_rag_index")

## 4. Ładowanie PLLuM

> Pobieranie ~16 GB wag zajmie 3–5 min. VRAM po załadowaniu: ok. 7 GB.

In [ ]:
import unsloth 
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="CYFRAGOVPL/Llama-PLLuM-8B-chat",
    max_seq_length=4096,
    load_in_4bit=True,     # QLoRA 4-bit
    dtype=None,
)
FastLanguageModel.for_inference(model)

free_vram = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
print(f"✅ Model załadowany. Wolne VRAM: {free_vram / 1024**3:.1f} GB")

## 5. Oficjalny PLLuM RAG prompt template

PLLuM był trenowany z **konkretnym formatem Jinja** — używamy go dokładnie.

In [8]:
def build_rag_prompt(question: str, docs: list[str]) -> str:
    """
    Oficjalny PLLuM RAG prompt template (Llama 3 format).
    Źródło: https://huggingface.co/CYFRAGOVPL/PLLuM-12B-base
    """
    doc_list = ""
    for i, doc in enumerate(docs):
        doc_list += f"Dokument: {i}\n{doc}\n"

    return (
        "<|begin_of_text|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        "Numerowana lista dokumentów jest poniżej:\n"
        "---------------------\n"
        f"<results>\n{doc_list}</results>\n"
        "---------------------\n"
        "Odpowiedz na pytanie użytkownika wykorzystując tylko informacje "
        "znajdujące się w dokumentach, a nie wcześniejszą wiedzę.\n"
        "Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi "
        "w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, "
        "z których pochodzą informacje. Zacytuj dokument za pomocą symbolu "
        "[nr_dokumentu] powołując się na fragment np. [0] dla fragmentu "
        "z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych "
        "do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "
        '"Nie udało mi się odnaleźć odpowiedzi na pytanie".\n\n'
        f"Pytanie: {question}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

print("✅ Funkcja build_rag_prompt gotowa.")

✅ Funkcja build_rag_prompt gotowa.


## 6. Pełny pipeline RAG

In [ ]:
def rag_query(
    question: str,
    k: int = 3,
    max_new_tokens: int = 512,
    temperature: float = 0.1,
    verbose: bool = False,
) -> str:
    """
    Pełny pipeline RAG:
    1. Wyszukuje k najbardziej podobnych fragmentów
    2. Buduje prompt PLLuM RAG template
    3. Generuje odpowiedź z cytatami

    Parametry:
        question      : pytanie użytkownika
        k             : liczba fragmentów (3–5 polecane)
        max_new_tokens: max długość odpowiedzi
        temperature   : 0.1 = faktyczne, 0.7 = kreatywne
        verbose       : True = pokaż pobrane fragmenty
    """
    # Wyszukuje podobne fragmenty
    results = vectorstore.similarity_search(question, k=k)
    retrieved_docs = [r.page_content for r in results]

    if verbose:
        print(f"=== Pobrane fragmenty (k={k}) ===")
        for i, doc in enumerate(retrieved_docs):
            print(f"[{i}] {doc[:200]}...\n")
        print("=" * 40)

    # Bbuduje prompt
    prompt = build_rag_prompt(question, retrieved_docs)

    # Generuje odpowiedź
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,  # redukuje powtarzanie fraz
        )

    # Dekoduje i wycina odpowiedź asystenta
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        return response.split("assistant")[-1].strip()
    return response.strip()


print("✅ Funkcja rag_query gotowa.")

✅ Funkcja rag_query gotowa.


## 7. Testowanie

In [ ]:
# Test 1: podstawowe pytanie
odpowiedz = rag_query(
    question="Ile wynoszą składki emerytalne?",
    k=3,
    verbose=True,   # pokaż pobrane fragmenty
)
print("\n🤖 Odpowiedź PLLuM:")
print(odpowiedz)

In [ ]:
# Test 2: pytanie wymagające syntezy z wielu fragmentów
odpowiedz = rag_query(
    question="W jakim terminie pracodawca musi zgłosić pracownika do ZUS?",
    k=3,
)
print("Odpowiedź:")
print(odpowiedz)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Odpowiedź:
Pracodawca musi zgłosić pracownika do ZUS w ciągu 7 dni od daty powstania obowiązku ubezpieczenia [0].


In [ ]:
# Test 3: pytanie spoza zakresu dokumentów
# Model powinien odpowiedzieć: "Nie udało mi się odnaleźć odpowiedzi"
odpowiedz = rag_query(
    question="Jaki jest kurs euro do złotego?",
)
print("Odpowiedź (brak w dokumentach):")
print(odpowiedz)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Odpowiedź (brak w dokumentach):
Nie udało mi się odnaleźć odpowiedzi na pytanie.


In [ ]:
# ===== TRYB INTERAKTYWNY =====
# Pytanie + Enter

while True:
    pytanie = input("\nTwoje pytanie (lub 'q' aby zakończyć): ").strip()
    if pytanie.lower() in ("q", "quit", "exit", ""):
        print("Do widzenia!")
        break
    odpowiedz = rag_query(pytanie, k=3)
    print(f"\n PLLuM: {odpowiedz}")

## 8. Diagnostyka i optymalizacja

Komórki pomocnicze — uruchamiaj w razie problemów.

In [ ]:
# Sprawdza zajętość VRAM
allocated = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM: {allocated:.2f} GB / {total:.2f} GB ({100*allocated/total:.1f}%)")

VRAM: 6.56 GB / 14.56 GB (45.1%)


In [ ]:
# Sprawdza jakość wyszukiwania — wyniki z podobieństwem (score)
def debug_search(question: str, k: int = 5):
    results = vectorstore.similarity_search_with_score(question, k=k)
    print(f"Pytanie: {question}\n")
    for i, (doc, score) in enumerate(results):
        print(f"[{i}] score={score:.4f} (niższy = lepszy dla FAISS L2)")
        print(f"     {doc.page_content[:150]}...\n")

debug_search("Ile wynosi zasiłek chorobowy?")

Pytanie: Ile wynosi zasiłek chorobowy?

[0] score=0.2287 (niższy = lepszy dla FAISS L2)
     Zasiłek chorobowy przysługuje ubezpieczonemu, który stał się niezdolny do pracy
    z powodu choroby w czasie trwania ubezpieczenia. Wynosi 80% podsta...

[1] score=0.3145 (niższy = lepszy dla FAISS L2)
     Składki na ubezpieczenie emerytalne wynoszą łącznie 19,52% podstawy wymiaru.
    Są one finansowane po połowie przez pracownika (9,76%) i pracodawcę (...

[2] score=0.3952 (niższy = lepszy dla FAISS L2)
     Emerytura z ZUS to świadczenie pieniężne mające służyć jako zabezpieczenie bytu
    na starość. Wiek emerytalny w Polsce wynosi 60 lat dla kobiet i 65...

[3] score=0.3969 (niższy = lepszy dla FAISS L2)
     ZUS (Zakład Ubezpieczeń Społecznych) jest państwową instytucją ubezpieczeń społecznych
    w Polsce. Zajmuje się gromadzeniem składek oraz wypłatą świ...

[4] score=0.3995 (niższy = lepszy dla FAISS L2)
     Płatnik składek ma obowiązek zgłosić pracownika do ubezpieczeń społecznych
